# Data exploration: groundwater recharge layers

Look at the processed layers, the AHP weights, the seasonal rainfall surfaces and the
resulting recharge zones. Run the pipeline first:

```bash
python -m backend.pipeline all
```

To explore the synthetic demo instead, set `DATA_DIR = "data/demo"` and `RESOLUTION = "250"` in the next cell
(after running `python scripts/make_synthetic_data.py` and the pipeline with the same settings).

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# Must be set before importing backend.config
DATA_DIR = "data"          # "data/demo" for the synthetic demo
RESOLUTION = "30"          # "250" for the synthetic demo
os.environ["GWR_DATA_DIR"] = str(ROOT / DATA_DIR)
os.environ["GWR_RESOLUTION"] = RESOLUTION

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from backend import config
from backend.ahp import default_result
from backend.grid import read_raster
from backend.overlay import available_layers

print("Data directory:", config.DATA_DIR)
print("Ranked layers available (annual):", available_layers("annual"))

## AHP weights and consistency

In [ ]:
res = default_result()
for c in res.criteria:
    print(f"{config.LAYER_LABELS[c]:<24s} {res.weights[c]:.3f}")
print(f"\nlambda_max = {res.lambda_max:.3f}, CI = {res.consistency_index:.4f}, CR = {res.consistency_ratio:.4f}")
print("Consistent:", res.is_consistent)

## Processed layers (raw values)

In [ ]:
def show(ax, arr, title, cmap="viridis"):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.7)

names = ["dem", "slope", "drainage_density", "lineament_density", "rainfall_annual", "lulc", "soil", "geology"]
names = [n for n in names if config.layer_path(n).exists()]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax in axes.flat:
    ax.axis("off")
for ax, n in zip(axes.flat, names):
    show(ax, read_raster(config.layer_path(n)), n.replace("_", " "))
plt.tight_layout()

## Ranked layers (1 = poor, 5 = excellent)

In [ ]:
rank_cmap = ListedColormap(["#ffffff"] + [config.RECHARGE_CLASSES[i]["color"] for i in range(1, 6)])
layers = available_layers("annual")
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax in axes.flat:
    ax.axis("off")
for ax, n in zip(axes.flat, layers):
    r = read_raster(config.ranked_path(n, "annual"), as_float=False)
    ax.imshow(r, cmap=rank_cmap, vmin=0, vmax=5, interpolation="nearest")
    counts = np.bincount(r.ravel(), minlength=6)[1:]
    ax.set_title(f"{config.LAYER_LABELS[n]}\nshare of ranks 1-5: " +
                 " ".join(f"{100 * c / max(counts.sum(), 1):.0f}%" for c in counts), fontsize=8)
plt.tight_layout()

## Seasonal rainfall (the temporal layer)

In [ ]:
seasons = [s for s in config.SEASONS if config.layer_path(f"rainfall_{s}").exists()]
fig, axes = plt.subplots(1, len(seasons), figsize=(4 * len(seasons), 4))
for ax, s in zip(np.atleast_1d(axes), seasons):
    r = read_raster(config.layer_path(f"rainfall_{s}"))
    im = ax.imshow(r, cmap="Blues")
    ax.set_title(f"{config.SEASONS[s]['label']}\n{np.nanmin(r):.0f}-{np.nanmax(r):.0f} mm", fontsize=9)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()

## Recharge zones and class areas per season

In [ ]:
stats_file = config.ZONES_DIR / "zone_statistics.json"
if stats_file.exists():
    stats = json.loads(stats_file.read_text())
    classes = [config.RECHARGE_CLASSES[i]["name"] for i in range(1, 6)]
    colors = [config.RECHARGE_CLASSES[i]["color"] for i in range(1, 6)]
    fig, ax = plt.subplots(figsize=(8, 4))
    bottom = np.zeros(len(stats))
    for i, (cls, col) in enumerate(zip(classes, colors)):
        vals = np.array([stats[s]["stats"][i]["percent"] for s in stats])
        ax.bar(list(stats), vals, bottom=bottom, color=col, label=cls, edgecolor="grey", linewidth=0.3)
        bottom += vals
    ax.set_ylabel("% of study area")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.set_title("Recharge class shares by season (same class breaks)")
    plt.tight_layout()
else:
    print("Run `python -m backend.pipeline overlay` first.")

In [ ]:
zone_cmap = ListedColormap(["#ffffff"] + [config.RECHARGE_CLASSES[i]["color"] for i in range(1, 6)])
seasons = [s for s in config.SEASONS if config.zones_path(s).exists()]
fig, axes = plt.subplots(1, len(seasons), figsize=(4 * len(seasons), 4))
for ax, s in zip(np.atleast_1d(axes), seasons):
    ax.imshow(read_raster(config.zones_path(s), as_float=False), cmap=zone_cmap, vmin=0, vmax=5,
              interpolation="nearest")
    ax.set_title(config.SEASONS[s]["label"], fontsize=9)
    ax.axis("off")
plt.tight_layout()

## Distribution of the recharge index (annual)

In [ ]:
gwrpi_file = config.ZONES_DIR / "gwrpi_annual.tif"
if gwrpi_file.exists():
    g = read_raster(gwrpi_file)
    breaks = json.loads((config.ZONES_DIR / "class_breaks.json").read_text())["breaks"]
    plt.figure(figsize=(8, 3.5))
    plt.hist(g[np.isfinite(g)], bins=80, color="#6a8caf")
    for b in breaks:
        plt.axvline(b, color="k", ls="--", lw=0.8)
    plt.xlabel("GWRPI (1-5)")
    plt.ylabel("pixels")
    plt.title("Annual GWRPI with class breaks")
    plt.tight_layout()